In [4]:
from tomobase import processes
from tomobase.data import Sinogram, Volume
import pathlib
import os
import stackview
import numpy as np

path = r"\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\TiltSeries128"
sample = 'Au1-ramp'

sino =  Sinogram.from_file(os.path.join(path, sample, 'start.h5'))
ramp = Sinogram.from_file(os.path.join(path, sample+'.mat'))


sino = processes.background_subtract_median(sino)
sino.data = np.transpose(sino.data, (0,2,1))
sino = processes.align_sinogram_xcorr(sino)
sino = processes.bin(sino, factor=8)
sino =  processes.normalize(sino)


(72, 1024, 1024) (72,) (72,)
(72, 1024, 1024) (72,) (72,)


100%|██████████| 72/72 [00:00<00:00, 285.51it/s]


In [ ]:
#delete first and last image
# import otsu
from skimage.filters import threshold_otsu

#sino.remove([63])
thresh = threshold_otsu(sino.data)
#sino.data[sino.data < thresh*0.6] = 0
sino = processes.align_sinogram_xcorr(sino)
#sino.data = np.roll(sino.data, shift=15, axis=2) # 2 is the x-axis
#crop_border = 64
#sino.data = sino.data[:, crop_border:-crop_border, crop_border:-crop_border]
print(sino.data.shape)
stackview.slice(sino.data)

In [ ]:
sino = processes.align_tilt_axis_shift(sino)
sino = processes.align_tilt_axis_rotation(sino)
stackview.slice(sino.data)

In [ ]:
sino.data = np.roll(sino.data, shift=-4, axis=2) # 2 is the x-axis
vol = processes.astra_reconstruct(sino, 'sirt', iterations=100)
vol.to_file(os.path.join(path, sample, 'start_recon.rec'))
stackview.orthogonal(vol.data)

In [ ]:
vol = Volume.from_file(os.path.join(path, sample, 'start_recon.rec'))
vol_compare = Volume.from_file(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Au1-ramp_morning-armadillo-72\0data.rec')
vol.data = np.transpose(vol.data, (2,1,0))
vol = processes.normalize(vol)
#vol.to_file(os.path.join(path, sample, 'end_recon_norm.rec'))
stackview.side_by_side(vol.data, vol_compare.data)

In [ ]:
vol = Volume.from_file(os.path.join(path, sample, 'start_recon.rec'))
vol = processes.normalize(vol)
vol.to_file(os.path.join(path, sample, 'end_recon_norm.rec'))

In [26]:

import tomobase 
import tomondt
import pathlib
import os 
import stackview
#Rod-D jumping-mountain-36 or olive-darkness-63
#Cage-D sage-forest-34
file_name = 'Au1-ramp'
instance_name = 'vital-smoke-40'
path = r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData'
#ref_path = pathlib.Path(os.path.join(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\Volumes128', file_name+'.vmf'))
dip_path = pathlib.Path(os.path.join(path,'DIP128', file_name+'_'+instance_name+'.vmf'))

print(dip_path)
#ref = tomondt.read_vmf(ref_path)
dip = tomondt.read_vmf(dip_path)



from skimage.metrics import structural_similarity as ssim
from skimage.filters import threshold_otsu

import numpy as np
from tomobase import processes
import stackview


min =10000
max = -10000
for t in dip.times:
    vol = dip.read_record(t)
    thresh = threshold_otsu(vol)
    vol = vol[vol >= thresh]

    if np.min(vol) < min:
        min = np.min(vol)
    if np.max(vol) > max:
        max = np.max(vol)


for t in dip.times:
    vol_dip = dip.read_record(t).transpose(0,2,1).copy() #.transpose(0,2,1)
    #vol_ref = ref.read_record(t)

def normalize_volume(vol, min, max):
    vol = (vol - min) / (max - min)
    return vol

def rmse(vol1, vol2):
    return np.sqrt(np.mean((vol1 - vol2) ** 2))*100

vol_dip = normalize_volume(vol_dip, min, max)
#vol_ref = normalize_volume(vol_ref, min, max)

print(max, min)
#stackview.side_by_side(vol_ref, vol_dip)

\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Au1-ramp_vital-smoke-40.vmf
4.303738594055176 1.2116482257843018


In [30]:
import os 
from tomobase.data import Volume

vol = Volume.from_file(os.path.join(path,'TiltSeries128', sample, 'end_recon.rec'))
vol_compare = Volume.from_file(os.path.join(path, 'DIP128', file_name+'_'+instance_name, '75data.rec'))
vol_compare.data = normalize_volume(vol_compare.data, min, max)

#vol_compare.data = np.transpose(vol_compare.data, (0,2,1))
vol = processes.normalize(vol)
vol_compare.to_file(os.path.join(path, 'TiltSeries128', sample, file_name+'_'+instance_name +'end_recon_norm.rec'))
vol.to_file(os.path.join(path, 'TiltSeries128', sample, 'end_recon_norm.rec'))
stackview.side_by_side(vol.data, vol_compare.data)

side_by_side
